# Loan Default Risk Analysis

**Dataset:** Lending Club Accepted Loans (2007–2018Q4), sampled to ~60,000 resolved loans (Fully Paid / Charged Off)

**Goal:** Clean the raw loan data, engineer a binary default target, and export it for SQL-based risk segmentation and a Power BI dashboard.

**Pipeline:** Python (this notebook) → SQLite (`queries.sql`) → Power BI dashboard

In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\anubh\Downloads\archive\accepted_2007_to_2018Q4.csv\accepted_2007_to_2018Q4.csv", low_memory=False)
df.shape

(2260701, 151)

In [2]:
df = df.sample(n=100000, random_state=42)
df.shape

(100000, 151)

In [3]:
df['loan_status'].value_counts()

loan_status
Fully Paid                                             47460
Current                                                38887
Charged Off                                            12034
Late (31-120 days)                                       969
In Grace Period                                          368
Late (16-30 days)                                        187
Does not meet the credit policy. Status:Fully Paid        60
Does not meet the credit policy. Status:Charged Off       31
Name: count, dtype: int64

In [4]:
# Cell 4
resolved_statuses = ['Fully Paid', 'Charged Off', 'Does not meet the credit policy. Status:Fully Paid', 'Does not meet the credit policy. Status:Charged Off']
df = df[df['loan_status'].isin(resolved_statuses)]
default_statuses = ['Charged Off', 'Does not meet the credit policy. Status:Charged Off']
df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)
df['is_default'].value_counts()

C:\Users\anubh\AppData\Local\Temp\ipykernel_7956\668159869.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['is_default'] = df['loan_status'].apply(lambda x: 1 if x in default_statuses else 0)


is_default
0    47520
1    12065
Name: count, dtype: int64

In [5]:
keep_cols = ['loan_amnt','funded_amnt','term','int_rate','installment','grade','sub_grade','emp_length','home_ownership','annual_inc','verification_status','issue_d','loan_status','purpose','addr_state','dti','delinq_2yrs','fico_range_low','fico_range_high','open_acc','pub_rec','revol_bal','revol_util','total_acc','is_default']
df = df[keep_cols]

In [6]:
df['emp_length'] = df['emp_length'].fillna('Unknown')
df['dti'] = df['dti'].fillna(df['dti'].median())
df['revol_util'] = df['revol_util'].fillna(df['revol_util'].median())

In [7]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['term'] = df['term'].str.strip().str.replace(' months', '').astype(int)

In [8]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.to_csv(r"C:\Users\anubh\Downloads\archive\loan_data_cleaned.csv", index=False)
print("Saved:", df.shape)

Saved: (59585, 25)


In [11]:
import sqlite3

conn = sqlite3.connect(r"C:\Users\anubh\Downloads\archive\loan_data.db")
df.to_sql('loans', conn, if_exists='replace', index=False)
conn.close()

print("Database created successfully")

Database created successfully
